# AI Data Analyst

**Project:** Natural-language questions over the `customer_shopping` MySQL database using Ollama (`qwen2.5-coder:7b`).

Flow: **Question → context resolution → SQL generation → validation → MySQL → result → natural-language answer → optional chart**.

## 1. Imports and configuration

Run this cell first.

In [2]:
import json
import re
import requests
import mysql.connector
import matplotlib.pyplot as plt
from getpass import getpass
from mysql.connector import Error

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "qwen2.5-coder:7b"
DB_NAME = "test"
TABLE_NAME = "customer_shopping"


## 2. Connect to MySQL

The cleaned/modelled data is already stored in MySQL, so this notebook does **not** repeat the earlier Pandas cleaning work.

In [1]:
try:
    # Connect to MySQL Server
    shaikdb = mysql.connector.connect(
        host="127.0.0.1",
        user="root",
        password="mysql1234"
    )

    if shaikdb.is_connected():
        print("Connected to MySQL successfully!")

    # Create a cursor
    cursor = shaikdb.cursor()

    # SQL query
    query = "CREATE DATABASE test"

    # Execute query
    cursor.execute(query)

    print("Database 'Test' created successfully!")

except Exception as e:
    print("Error:", e)

finally:
    if 'shaikdb' in locals() and shaikdb.is_connected():
        cursor.close()
        shaikdb.close()
        print("Connection closed.")

Error: name 'mysql' is not defined


## 3. Load schema and allowed categorical values

These values are supplied to the SQL model so it does not invent column names or category values.

In [ ]:
cursor.execute(f"DESCRIBE {TABLE_NAME}")
columns = cursor.fetchall()

schema = "\n".join(
    f"- {column[0]} ({column[1]})"
    for column in columns
)

print(schema)

categorical_columns = [
    "gender", "category", "location", "size", "season",
    "subscription_status", "shipping_type", "discount_applied",
    "payment_method", "frequency_of_purchases", "age_group"
]

categorical_values = {}

for column in categorical_columns:
    cursor.execute(f"SELECT DISTINCT `{column}` FROM {TABLE_NAME}")
    categorical_values[column] = [row[0] for row in cursor.fetchall()]

print("\nCategorical values loaded for", len(categorical_values), "columns.")


## 4. Conversation context resolver

Only the most recent standalone question is used as context. This avoids older questions incorrectly overriding the latest follow-up.

In [ ]:
conversation_history = []

def resolve_context(question):
    if not conversation_history:
        return question.strip()

    previous = conversation_history[-1]
    previous_question = previous.get("standalone_question", previous.get("question", ""))

    context_prompt = f"""
You are a conversational AI data analyst.

Your ONLY job is to rewrite the CURRENT USER QUESTION
into one complete standalone data question.

Do NOT answer the question.
Do NOT generate SQL.
Do NOT explain anything.

PREVIOUS STANDALONE QUESTION:
{previous_question}

CURRENT USER QUESTION:
{question}

RULES:
1. The previous standalone question is the main context.
2. If the current question is standalone, return it unchanged.
3. If it is a follow-up, preserve intent, metric, target, and filters unless explicitly changed.
4. If the user says "What about...", change only the explicitly changed condition.
5. If the user says "How many are there?", preserve the subject and filters.
6. The most recent question always has priority.
7. Do not go back to older questions when the previous standalone question already contains the needed context.
8. Preserve negative conditions such as "not subscribed", "No", "isn't", and "aren't".
9. Do not invent new conditions.
10. Return ONLY the standalone question.

STANDALONE QUESTION:
"""

    try:
        response = requests.post(
            OLLAMA_URL,
            json={
                "model": MODEL,
                "prompt": context_prompt,
                "stream": False,
                "options": {"temperature": 0}
            },
            timeout=60
        )
        response.raise_for_status()
        data = response.json()
        standalone_question = data.get("response", "").strip()

        if not standalone_question:
            return question.strip()

        return standalone_question.replace("```", "").strip().strip('"').strip("'")

    except requests.exceptions.ConnectionError:
        print("Could not connect to Ollama.")
    except requests.exceptions.Timeout:
        print("Context resolution took too long.")
    except requests.exceptions.RequestException as e:
        print("Context resolver error:", e)

    return question.strip()


## 5. SQL safety validator

Only one `SELECT` statement against `customer_shopping` is allowed.

In [ ]:
def validate_sql(sql):
    sql_clean = sql.strip()

    if not sql_clean:
        return False, "SQL query is empty."

    if sql_clean.endswith(";"):
        sql_clean = sql_clean[:-1].strip()

    if ";" in sql_clean:
        return False, "Multiple SQL statements are not allowed."

    if not sql_clean.lower().startswith("select"):
        return False, "Only SELECT queries are allowed."

    forbidden = [
        "insert", "update", "delete", "drop", "alter",
        "truncate", "create", "replace", "grant", "revoke"
    ]

    sql_lower = sql_clean.lower()

    for keyword in forbidden:
        if re.search(rf"\b{keyword}\b", sql_lower):
            return False, f"Forbidden SQL operation detected: {keyword.upper()}"

    tables = re.findall(r"\bfrom\s+([a-zA-Z0-9_]+)", sql_lower)

    for table in tables:
        if table != TABLE_NAME:
            return False, f"Access to table '{table}' is not allowed."

    return True, "SQL is safe to execute."


## 6. Result formatting and chart helper

In [ ]:
def format_result(cursor, result):
    column_names = [description[0] for description in cursor.description]

    formatted_result = [
        dict(zip(column_names, row))
        for row in result
    ]

    return [
        {
            key: float(value) if hasattr(value, "as_tuple") else value
            for key, value in row.items()
        }
        for row in formatted_result
    ]


def create_chart(formatted_result):
    if len(formatted_result) < 2:
        return False

    columns = list(formatted_result[0].keys())
    text_columns = []
    numeric_columns = []

    for column in columns:
        value = formatted_result[0][column]
        if isinstance(value, (int, float)):
            numeric_columns.append(column)
        else:
            text_columns.append(column)

    if len(text_columns) != 1 or len(numeric_columns) != 1:
        return False

    category_column = text_columns[0]
    value_column = numeric_columns[0]

    categories = [row[category_column] for row in formatted_result]
    values = [row[value_column] for row in formatted_result]

    plt.figure(figsize=(8, 5))
    plt.bar(categories, values)
    plt.title(
        f"{value_column.replace('_', ' ').title()} by "
        f"{category_column.replace('_', ' ').title()}"
    )
    plt.xlabel(category_column.replace("_", " ").title())
    plt.ylabel(value_column.replace("_", " ").title())
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

    return True


## 7. Main AI Data Analyst

The SQL-generation model first decides whether the question is supported. Unsupported questions stop before MySQL execution.

In [ ]:
def ask_data_analyst():
    question = input("Ask your data question: ").strip()

    if not question:
        print("Please enter a question.")
        return

    standalone_question = resolve_context(question)

    print("\nInterpreted Question:")
    print(standalone_question)

    prompt = f"""
You are a MySQL SQL generator for an AI data analyst.

Your job is to determine whether the user's question can be answered using the available database and, if supported, generate ONE correct MySQL SELECT query.

DATABASE: test
TABLE: customer_shopping

SCHEMA:
{schema}

ALLOWED CATEGORICAL VALUES:
"""

    for column, values in categorical_values.items():
        prompt += f"\n{column}: {', '.join(map(str, values))}"

    prompt += f"""

IMPORTANT RULES:
1. Understand the user's question carefully.
2. Use ONLY information explicitly requested.
3. Do not add unrequested filters, conditions, columns, GROUP BY, or ORDER BY clauses.
4. Use AVG() for averages, SUM() for totals, and COUNT() for counts.
5. Use ORDER BY and LIMIT when highest/lowest or top/bottom is requested.
6. Never assume additional conditions.
7. Use only the provided table and columns.
8. Use only the listed categorical values.
9. Generate exactly ONE SELECT query.
10. Never modify the database.
11. Return ONLY valid JSON.
12. The JSON must contain exactly: sql, supported, reason.
13. Do not use markdown.
14. Do not add explanations outside JSON.
15. Never invent columns, tables, values, or information.
16. If the question cannot be answered from this database, do not generate SQL.

SUPPORTED FORMAT:
{{
  "sql": "SELECT ...",
  "supported": true,
  "reason": ""
}}

UNSUPPORTED FORMAT:
{{
  "sql": null,
  "supported": false,
  "reason": "The question cannot be answered using the available database."
}}

USER QUESTION:
{standalone_question}
"""

    try:
        response = requests.post(
            OLLAMA_URL,
            json={
                "model": MODEL,
                "prompt": prompt,
                "stream": False,
                "options": {"temperature": 0}
            },
            timeout=60
        )
        response.raise_for_status()
        data = response.json()
        raw_response = data.get("response", "").strip()

    except requests.exceptions.ConnectionError:
        print("Could not connect to Ollama. Make sure Ollama is running.")
        return
    except requests.exceptions.Timeout:
        print("The AI request took too long.")
        return
    except requests.exceptions.RequestException as e:
        print("AI request error:", e)
        return

    if not raw_response:
        print("The AI returned an empty response.")
        return

    try:
        structured_data = json.loads(raw_response)
    except json.JSONDecodeError:
        print("\nAI returned invalid JSON. SQL was not executed.")
        return

    if not isinstance(structured_data, dict):
        print("\nAI response is not a JSON object.")
        return

    required = {"sql", "supported", "reason"}
    if set(structured_data.keys()) != required:
        print("\nAI response has an invalid JSON structure.")
        return

    supported = structured_data["supported"]
    sql = structured_data["sql"]
    reason = structured_data["reason"]

    if not isinstance(supported, bool) or not isinstance(reason, str):
        print("\nInvalid supported/reason values in AI response.")
        return

    if supported is False:
        print("\nQuestion not supported.")
        print("Reason:", reason)
        return

    if not isinstance(sql, str) or not sql.strip():
        print("\nSupported question returned no SQL query.")
        return

    sql = sql.strip()

    is_valid, message = validate_sql(sql)

    print("\nGenerated SQL:")
    print(sql)
    print("\nValidation:")
    print(message)

    if not is_valid:
        print("Query was blocked.")
        return

    try:
        cursor.execute(sql)
        result = cursor.fetchall()
    except Error as e:
        print("Database error:", e)
        return

    if not result or result == [(None,)]:
        print("No matching data was found.")
        return

    formatted_result = format_result(cursor, result)

    print("\nFormatted Result:")
    print(formatted_result)

    answer_prompt = f"""
You are a concise AI data analyst.

Answer the user's question using ONLY the database result.

USER QUESTION:
{standalone_question}

DATABASE RESULT:
{formatted_result}

RULES:
1. Use only the database result.
2. Do not invent information or make assumptions.
3. Keep the answer concise and easy to understand.
4. Clearly mention the relevant metric, category, filter, or condition when appropriate.
5. If there are multiple results, present them clearly.
6. Do not mention SQL, prompts, the model, or internal processing.
7. Return ONLY the natural-language answer.
"""

    try:
        answer_response = requests.post(
            OLLAMA_URL,
            json={
                "model": MODEL,
                "prompt": answer_prompt,
                "stream": False,
                "options": {"temperature": 0.1}
            },
            timeout=60
        )
        answer_response.raise_for_status()
        answer_data = answer_response.json()
        answer = answer_data.get("response", "").strip()

    except requests.exceptions.ConnectionError:
        print("Could not connect to Ollama.")
        return
    except requests.exceptions.Timeout:
        print("The answer-generation request took too long.")
        return
    except requests.exceptions.RequestException as e:
        print("AI request error:", e)
        return

    if not answer:
        print("The answer generator returned an empty response.")
        return

    print("\nAI Answer:")
    print(answer)

    if create_chart(formatted_result):
        print("\nChart generated automatically.")

    conversation_history.append({
        "question": question,
        "standalone_question": standalone_question,
        "sql": sql,
        "result": formatted_result,
        "answer": answer
    })


## 8. Test the pipeline

Start with a supported question, then test an unsupported question.

In [ ]:
# Supported test
ask_data_analyst()

### Useful test questions

- `How many customers are there?`
- `What is the average purchase amount?`
- `What is the average purchase amount for subscribed customers?`
- `What about customers who are not subscribed?`
- `How many are there?`
- `What is the total purchase amount for each category?`
- `Who is the CEO of Apple?` → should be rejected as unsupported.

## 9. Reset conversation history

Run this only when you want to start a fresh conversation.

In [ ]:
conversation_history = []
print("Conversation history cleared.")

## Project status

The notebook now contains only the production-relevant pipeline. Earlier exploratory EDA, repeated SQL experiments, duplicate Ollama calls, duplicate answer-generation cells, test outputs, and old versions of `ask_data_analyst()` have been removed.